In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS zordering.silver;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Read bronze tables

departments_df = spark.table("zordering.bronze.departments_raw")

employee_df = spark.table("zordering.bronze.employee_raw")

salaries_df = spark.table("zordering.bronze.salaries_raw")


# Join all tables

joined_df = (
    employee_df
    .join(departments_df, on="department_name", how="left")
    .join(salaries_df, on="employee_id", how="left")
)


# 1. Remove null first_name and last_name records

clean_df = joined_df.filter(
    col("first_name").isNotNull() &
    col("last_name").isNotNull()
)


# 2. Fill null salary with 70% of average salary

avg_salary = clean_df.select(avg("base_salary")).collect()[0][0]

clean_df = clean_df.withColumn(
    "base_salary",
    when(
        col("base_salary").isNull(),
        lit(avg_salary * 0.7)
    ).otherwise(col("base_salary"))
)


# 3. Department-wise average salary

dept_window = Window.partitionBy("department_name")

silver_df = clean_df.withColumn(
    "dept_avg_salary",
    avg("base_salary").over(dept_window)
)


# Bonus calculation

silver_df = silver_df.withColumn(
    "bonus_amount",
    when(
        col("base_salary") > col("dept_avg_salary"),
        col("base_salary") * 0.10
    ).otherwise(
        col("base_salary") * 0.20
    )
)


# Final salary

silver_df = silver_df.withColumn(
    "final_salary",
    col("base_salary") + col("bonus_amount")
)


# Remove helper column

silver_df = silver_df.drop("dept_avg_salary")


# Save to silver layer

silver_df.write.mode("overwrite").saveAsTable(
    "zordering.silver.employee_salary_summary"
)

In [0]:
%sql
SELECT * FROM zordering.silver.employee_salary_summary;